# YouTube Data Collection — NASCAR Race Video Engagement

Collects race highlight video statistics using the YouTube Data API v3.
Requires `YOUTUBE_API_KEY` in `config.py`.

**Daily quota:** 10,000 units. This full run costs ~4,000 units.

## Step 1: Test YouTube Connection

In [1]:
from googleapiclient.discovery import build
from config import YOUTUBE_API_KEY
import pandas as pd
from datetime import datetime, timedelta
import time
import json

# Initialize YouTube API client
youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)

# Test connection with a simple search
test_request = youtube.search().list(
    q="NASCAR 2024 highlights",
    part="snippet",
    maxResults=5,
    type="video"
)

test_response = test_request.execute()

print("YouTube API connected successfully!")
print(f"Found {len(test_response['items'])} videos")
print("\nSample results:")
for item in test_response['items']:
    print(f"  - {item['snippet']['title'][:60]}...")

YouTube API connected successfully!
Found 5 videos

Sample results:
  - 2024 Daytona 500 Highlights | NASCAR on FOX...
  - NASCAR Cup Series EXTENDED HIGHLIGHTS: Championship race at ...
  - NASCAR Cup Series EXTENDED HIGHLIGHTS: Straight Talk Wireles...
  - 2026 Coca-Cola 600 | NASCAR Cup Series Highlights...
  - NASCAR Cup Series Highlights | 2026 Kansas Speedway...


## Step 2: Load Race Schedule & Define Sponsor-Driver Associations

In [2]:
# Load race schedule from Module 2.1 processed data
race_df = pd.read_csv('../data/processed/race_results_2024_clean.csv')

# Add race dates (2024 NASCAR Cup Series schedule — approximate)
season_start = datetime(2024, 2, 18)
season_end   = datetime(2024, 11, 10)
total_days   = (season_end - season_start).days
max_race     = race_df['Race_Number'].max()

race_dates_2024 = {
    r: (season_start + timedelta(days=total_days * (r - 1) / (max_race - 1))).strftime('%Y-%m-%d')
    for r in range(1, max_race + 1)
}

race_df['Race_Date'] = pd.to_datetime(race_df['Race_Number'].map(race_dates_2024))

# Build race list (one row per unique race)
race_list = (
    race_df.groupby(['Race_Number', 'Race_Name'])
    .agg(Race_Date=('Race_Date', 'first'))
    .reset_index()
    .to_dict('records')
)

print(f"Loaded {len(race_list)} races")

# Sponsor-driver associations for video tagging
sponsor_drivers = {
    'FedEx': ['Hamlin', 'Denny Hamlin'],
    'NAPA Auto Parts': ['Elliott', 'Chase Elliott'],
    "McDonald's": ['Wallace', 'Bubba Wallace'],
    "Love's Travel Stops": ['McDowell', 'Michael McDowell'],
}

Loaded 38 races


## Step 3: Build the Race Video Search

In [3]:
def search_race_videos(youtube, race_name, track_name=None, max_results=10):
    """
    Search YouTube for videos related to a specific NASCAR race.

    Parameters:
    -----------
    youtube : googleapiclient.discovery.Resource
        Authenticated YouTube API client
    race_name : str
        Official race name
    track_name : str
        Track/location name (optional, improves search relevance)
    max_results : int
        Maximum videos to retrieve

    Returns:
    --------
    list : List of video dictionaries
    """
    # Construct search query
    query = f"NASCAR {race_name} 2024 highlights"
    if track_name:
        query = f"NASCAR {race_name} {track_name} 2024"

    try:
        search_response = youtube.search().list(
            q=query,
            part="snippet",
            maxResults=max_results,
            type="video",
            videoDuration="medium",  # Filter to medium length videos
            order="viewCount"        # Get most viewed first
        ).execute()

        videos = []
        for item in search_response['items']:
            videos.append({
                'video_id': item['id']['videoId'],
                'title': item['snippet']['title'],
                'channel': item['snippet']['channelTitle'],
                'publish_date': item['snippet']['publishedAt'],
                'description': item['snippet'].get('description', '')[:200]
            })

        return videos

    except Exception as e:
        print(f"  Error searching: {e}")
        return []

In [4]:
def get_video_statistics(youtube, video_ids):
    """
    Get detailed statistics for a list of videos.

    Parameters:
    -----------
    youtube : googleapiclient.discovery.Resource
        Authenticated YouTube API client
    video_ids : list
        List of YouTube video IDs

    Returns:
    --------
    dict : Dictionary mapping video_id to statistics
    """
    if not video_ids:
        return {}

    # YouTube API allows up to 50 videos per request
    stats = {}

    # Process in batches of 50
    for i in range(0, len(video_ids), 50):
        batch = video_ids[i:i + 50]

        try:
            response = youtube.videos().list(
                part="statistics,contentDetails",
                id=",".join(batch)
            ).execute()

            for item in response['items']:
                video_id  = item['id']
                statistics = item.get('statistics', {})

                stats[video_id] = {
                    'view_count':    int(statistics.get('viewCount', 0)),
                    'like_count':    int(statistics.get('likeCount', 0)),
                    'comment_count': int(statistics.get('commentCount', 0)),
                    'duration':      item['contentDetails'].get('duration', '')
                }

        except Exception as e:
            print(f"  Error getting stats: {e}")

    return stats

## Step 4: Collect Race Video Data

In [5]:
def collect_race_videos(youtube, races, videos_per_race=10):
    """
    Collect YouTube video data for all races in the season.

    Parameters:
    -----------
    youtube : googleapiclient.discovery.Resource
        Authenticated YouTube API client
    races : list
        List of race dictionaries with Race_Number, Race_Name, Race_Date
    videos_per_race : int
        Maximum videos to collect per race

    Returns:
    --------
    pd.DataFrame : Combined video data for all races
    """
    all_videos = []

    for i, race in enumerate(races, 1):
        race_name = race.get('Race_Name', race.get('name', ''))
        race_date = race.get('Race_Date', race.get('date', ''))
        race_num  = race.get('Race_Number', race.get('race_number', i))

        print(f"\n[{i}/{len(races)}] Searching: {race_name}")

        # Search for race videos
        videos = search_race_videos(
            youtube,
            race_name,
            max_results=videos_per_race
        )

        if videos:
            # Get statistics for all videos
            video_ids = [v['video_id'] for v in videos]
            stats = get_video_statistics(youtube, video_ids)

            # Combine search results with statistics
            for video in videos:
                video_id   = video['video_id']
                video_stats = stats.get(video_id, {})

                all_videos.append({
                    'race_number':   race_num,
                    'race_name':     race_name,
                    'race_date':     str(race_date),
                    'video_id':      video_id,
                    'video_title':   video['title'],
                    'channel':       video['channel'],
                    'publish_date':  video['publish_date'],
                    'view_count':    video_stats.get('view_count', 0),
                    'like_count':    video_stats.get('like_count', 0),
                    'comment_count': video_stats.get('comment_count', 0),
                    'duration':      video_stats.get('duration', ''),
                    'url':           f"https://youtube.com/watch?v={video_id}"
                })

            total_views = sum(s.get('view_count', 0) for s in stats.values())
            print(f"  Found {len(videos)} videos, total views: {total_views:,}")
        else:
            print(f"  No videos found")

        # Rate limiting
        time.sleep(1)

    return pd.DataFrame(all_videos)


# Collect videos for all races
# Note: This uses API quota — monitor your usage
print("Starting YouTube video collection...")
print("=" * 50)

youtube_df = collect_race_videos(youtube, race_list, videos_per_race=10)

print("\n" + "=" * 50)
print(f"Collection complete!")
print(f"Total videos collected: {len(youtube_df)}")
print(f"Total views tracked: {youtube_df['view_count'].sum():,}")

Starting YouTube video collection...

[1/38] Searching: Daytona
  Found 10 videos, total views: 30,190,028

[2/38] Searching: Daytona
  Found 10 videos, total views: 30,190,060

[3/38] Searching: Daytona
  Found 10 videos, total views: 30,190,060

[4/38] Searching: Atlanta
  Found 10 videos, total views: 6,529,984

[5/38] Searching: Las Vegas
  Found 10 videos, total views: 43,875,197

[6/38] Searching: Phoenix
  Found 10 videos, total views: 1,842,525

[7/38] Searching: Bristol
  Found 10 videos, total views: 5,188,980

[8/38] Searching: COTA
  Found 10 videos, total views: 11,244,309

[9/38] Searching: Richmond
  Found 10 videos, total views: 1,772,211

[10/38] Searching: Martinsville
  Found 10 videos, total views: 1,951,523

[11/38] Searching: Texas
  Found 10 videos, total views: 11,059,863

[12/38] Searching: Talladega
  Found 10 videos, total views: 17,485,683

[13/38] Searching: Dover
  Found 10 videos, total views: 1,779,069

[14/38] Searching: Kansas
  Found 10 videos, total 

## Step 5: Identify Sponsor-Relevant Videos

In [6]:
def identify_sponsor_videos(df, sponsor_drivers):
    """
    Tag videos that likely feature specific sponsors based on title/description.
    """
    df = df.copy()
    for sponsor, drivers in sponsor_drivers.items():
        # Check if any driver name appears in title
        pattern = '|'.join(drivers)
        df[f'features_{sponsor}'] = df['video_title'].str.contains(
            pattern, case=False, na=False
        )
    return df


# Tag videos with sponsor associations
youtube_df = identify_sponsor_videos(youtube_df, sponsor_drivers)

# Preview tagged results
print("Videos featuring target sponsors:")
for sponsor in sponsor_drivers.keys():
    col   = f'features_{sponsor}'
    count = youtube_df[col].sum() if col in youtube_df.columns else 0
    print(f"  {sponsor}: {count} videos")

Videos featuring target sponsors:
  FedEx: 0 videos
  NAPA Auto Parts: 0 videos
  McDonald's: 9 videos
  Love's Travel Stops: 0 videos


In [7]:
def aggregate_sponsor_exposure(df, sponsor_drivers):
    """
    Aggregate YouTube exposure by sponsor and race.
    """
    results = []

    for race_name in df['race_name'].unique():
        race_df   = df[df['race_name'] == race_name]
        race_num  = race_df['race_number'].iloc[0]
        race_date = race_df['race_date'].iloc[0]

        # Total race exposure (all videos)
        total_views = race_df['view_count'].sum()

        for sponsor in sponsor_drivers.keys():
            feature_col    = f'features_{sponsor}'
            sponsor_videos = race_df[race_df[feature_col] == True] if feature_col in race_df.columns else race_df.iloc[0:0]

            results.append({
                'race_number':         race_num,
                'race_name':           race_name,
                'race_date':           race_date,
                'sponsor':             sponsor,
                'total_race_views':    total_views,
                'sponsor_video_count': len(sponsor_videos),
                'sponsor_video_views': sponsor_videos['view_count'].sum(),
                'sponsor_video_likes': sponsor_videos['like_count'].sum(),
                'view_share': (
                    sponsor_videos['view_count'].sum() / total_views * 100
                    if total_views > 0 else 0
                )
            })

    return pd.DataFrame(results)


youtube_sponsor_summary = aggregate_sponsor_exposure(youtube_df, sponsor_drivers)

print("\nSponsor YouTube Exposure Summary (sample):")
print(youtube_sponsor_summary.head(15))


Sponsor YouTube Exposure Summary (sample):
    race_number  race_name            race_date              sponsor  \
0            -2    Daytona                  NaT                FedEx   
1            -2    Daytona                  NaT      NAPA Auto Parts   
2            -2    Daytona                  NaT           McDonald's   
3            -2    Daytona                  NaT  Love's Travel Stops   
4             2    Atlanta  2024-02-25 00:00:00                FedEx   
5             2    Atlanta  2024-02-25 00:00:00      NAPA Auto Parts   
6             2    Atlanta  2024-02-25 00:00:00           McDonald's   
7             2    Atlanta  2024-02-25 00:00:00  Love's Travel Stops   
8             3  Las Vegas  2024-03-04 00:00:00                FedEx   
9             3  Las Vegas  2024-03-04 00:00:00      NAPA Auto Parts   
10            3  Las Vegas  2024-03-04 00:00:00           McDonald's   
11            3  Las Vegas  2024-03-04 00:00:00  Love's Travel Stops   
12            4    P

## Step 6: Save YouTube Data

In [8]:
# Save raw video data
raw_output = '../data/raw/youtube_videos_2024_raw.csv'
youtube_df.to_csv(raw_output, index=False)
print(f"Saved raw video data to: {raw_output}")

# Save sponsor exposure summary
summary_output = '../data/processed/youtube_engagement_2024.csv'
youtube_sponsor_summary.to_csv(summary_output, index=False)
print(f"Saved sponsor summary to: {summary_output}")

# Also save as final deliverable name
youtube_sponsor_summary.to_csv('../data/processed/youtube_engagement.csv', index=False)
print("Saved final deliverable: youtube_engagement.csv")

# Save collection metadata
youtube_metadata = {
    'collection_date':  datetime.now().isoformat(),
    'total_videos':     len(youtube_df),
    'total_views':      int(youtube_df['view_count'].sum()),
    'races_covered':    youtube_df['race_name'].nunique(),
    'sponsors_tracked': list(sponsor_drivers.keys())
}

with open('../data/processed/youtube_collection_metadata.json', 'w') as f:
    json.dump(youtube_metadata, f, indent=2)
print("Saved collection metadata")

print("\n=== YouTube Data Collection Complete ===")

Saved raw video data to: ../data/raw/youtube_videos_2024_raw.csv
Saved sponsor summary to: ../data/processed/youtube_engagement_2024.csv
Saved final deliverable: youtube_engagement.csv
Saved collection metadata

=== YouTube Data Collection Complete ===
